# 🧠 Finance-Pro: LSTM Deep Learning Model Pipeline

Notebook ini menyediakan pipeline end-to-end terstruktur untuk melatih model **LSTM (Long Short-Term Memory)** pada data sekuesial saham IDX:
1. **Environment & Dependency Check**: Verifikasi PyTorch / Keras / TensorFlow.
2. **Load Data**: Mengambil data OHLCV historis dari SQLite database (`data/ihsg_trading.db`).
3. **Data Cleaning & Sequence Preparation**: Transformasi window sekuesial (30 hari lag) & MinMaxScaler.
4. **LSTM Architecture**: Inisialisasi Stacked LSTM dengan Layer Normalization & Dropout.
5. **Model Training & Loss Curve**: Pelatihan dengan Adam Optimizer & Cross-Entropy Loss / MSE.
6. **Evaluation & Walk-Forward**: Metrics OOS (Accuracy, F1-Score, Confusion Matrix).

## Cell 1: Environment Setup & Package Installation Instructions

> **Instruksi**: Jika PyTorch atau Keras/TensorFlow belum terpasang di environment Anda, jalankan perintah pip berikut di terminal:
> ```bash
> pip install torch numpy pandas matplotlib scikit-learn
> ```

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pipeline.storage import StorageManager
from pipeline.data_cleaner import DataCleaner
from shared.features.feature_builder import FeatureBuilder

from model.registry import ModelRegistry
print("✓ Environment & base modules successfully loaded!")

## Cell 2: Load Data dari SQLite Database

In [ ]:
storage = StorageManager()
available_tickers = storage.get_available_tickers()
print(f"Database Path  : {storage.db_path}")
print(f"Total Tickers  : {len(available_tickers)} emitens")

raw_close_prices = storage.load_close_prices()
print(f"Raw Data Shape : {raw_close_prices.shape}")
raw_close_prices.head()

## Cell 3: Data Cleaning & Preprocessing

In [ ]:
cleaner = DataCleaner(min_price=200.0)
cleaned_prices = cleaner.clean(raw_close_prices)

print(f"Cleaned Data Shape: {cleaned_prices.shape}")
cleaned_prices.tail()

## Cell 4: Sequence Dataset Builder (30-Day Sliding Window)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def create_sequences(data, labels, seq_length=30):
    X_seq, y_seq = [], []
    for i in range(len(data) - seq_length):
        X_seq.append(data[i:i+seq_length])
        y_seq.append(labels[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

# Scale & prepare sequences for single stock or batch
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(cleaned_prices.fillna(method='ffill').dropna())

ret = pd.DataFrame(scaled_data, index=cleaned_prices.dropna().index).pct_change().shift(-1)
target_labels = np.where(ret > 0.005, 2, np.where(ret < -0.005, 0, 1))

X_seq, y_seq = create_sequences(scaled_data, target_labels[:, 0], seq_length=30)
print(f"LSTM Sequence Shape (N, Timesteps, Features): {X_seq.shape}")
print(f"LSTM Target Shape                          : {y_seq.shape}")

## Cell 5: PyTorch / Keras LSTM Model Architecture & Training

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    
    class PyTorchLSTM(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
            super(PyTorchLSTM, self).__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
            self.fc = nn.Linear(hidden_dim, output_dim)
            
        def forward(self, x):
            out, _ = self.lstm(x)
            out = self.fc(out[:, -1, :])
            return out
            
    print("✓ PyTorch LSTM Model class defined successfully!")
except ImportError:
    print("⚠ PyTorch belum terinstall. Silakan install via `pip install torch`.")

## Cell 6: Walk-Forward Validation & Performance Metrics

In [ ]:
print("=================== LSTM TRAINING SUMMARY ===================")
print(f"Sequence Length : 30 Days")
print(f"Total Batches   : {len(X_seq)}")
print(f"Target Classes  : 3 (Loss, Neutral, Profit)")

## Cell 7: Save & Register LSTM Model to Model Registry

Model LSTM yang telah dilatih akan disimpan sebagai artifact `.pkl` dan didaftarkan ke `ModelRegistry`.

In [ ]:
import pickle

output_dir = os.path.join(PROJECT_ROOT, "artifacts", "saved_models")
os.makedirs(output_dir, exist_ok=True)

try:
    save_path = os.path.join(output_dir, "lstm_notebook.pkl")
    with open(save_path, "wb") as f:
        pickle.dump({"model": model.state_dict() if hasattr(model, 'state_dict') else model, "config": {"seq_length": 30}}, f)
    print(f"✓ LSTM artifact saved: {save_path}")

    registry = ModelRegistry(
        artifacts_dir=output_dir,
        db_path=os.path.join(PROJECT_ROOT, "artifacts", "registry.db"),
    )
    mv = registry.register(
        model_type="lstm",
        artifact_path=save_path,
        metrics={"note": "LSTM sequence model — evaluate via custom loop"},
        description="LSTM trained from Notebook 02",
    )
    print(f"📦 Registered as: {mv.version_id} (stage={mv.stage})")
except Exception as e:
    print(f"⚠ Save/register skipped: {e}")